In [ ]:
import pandas as pd

df = pd.read_csv("2026-10-15_daily_metrics.csv")
df = df[df["日付"] != "合計"]   # 2行目の「合計」行を除く
df["日付"] = pd.to_datetime(df["日付"])
df = df.sort_values("日付")
print(len(df), df["視聴回数"].sum())   # 91行・36,577回になるか

df["ma7"] = df["視聴回数"].rolling(window=7).mean()
print(df.loc[df["日付"] == "2026-10-14", "ma7"].round(2))   # 602.29になるか

In [ ]:
# 最初の7日：pandasのrolling(7)は6日目まで空（NaN）。min_periods=1でSQLのROWS BETWEEN 6 PRECEDINGと同じになる
df["ma7_sql"] = df["視聴回数"].rolling(window=7, min_periods=1).mean()
print(df[["日付", "視聴回数", "ma7", "ma7_sql"]].head(8))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 4))
plt.plot(df["日付"], df["視聴回数"], label="views")
plt.plot(df["日付"], df["ma7"], label="ma7")
plt.legend(); plt.show()

In [ ]:
# Shorts と長尺の比較（SQL の v_video と同じ分け方）
v = pd.read_csv("2026-10-15_videos.csv")
v = v[v["コンテンツ"] != "合計"].copy()   # 2行目の「合計」行を除く
v["視聴回数"] = v["視聴回数"].fillna(0)   # 空欄はSQLのISNULL(…, 0)と同じく0にする
v["公開日"] = pd.to_datetime(v["動画公開時刻"], format="%b %d, %Y")
v["種類"] = "Shorts"
v.loc[v["長さ"] > 60, "種類"] = "長尺"
v.loc[(v["長さ"] == 0) | v["動画のタイトル"].str.contains("LIVE"), "種類"] = "Live"
print(len(v), v["視聴回数"].sum())   # 342本・36,474回になるか
print(v.groupby("種類")["視聴回数"].agg(["count", "sum", "mean"]).round(1))   # Shorts 201本・157.4回／長尺 138本・32.4回になるか

In [ ]:
# 公開月ごとの1本あたり視聴回数（長尺 72.2→31.4→20.9→4.5、Shorts 79.7→178.2 になるか）
v["公開月"] = v["公開日"].dt.strftime("%Y-%m")
by_month = v[v["種類"] != "Live"].pivot_table(index="公開月", columns="種類", values="視聴回数", aggfunc="mean").round(1)
print(by_month)
chart = by_month.rename(columns={"長尺": "Long"}).rename_axis(index="Publish month", columns="Type")   # グラフは日本語が文字化けするので英語にする
chart.plot(kind="bar", figsize=(8, 4), rot=0, title="Views per video by publish month")
plt.show()

In [ ]:
# 公開から7日間（公開日0〜6日目）の伸び：種類ごとの上位5本（SQL クエリ③'・Power BI 3ページ目と同じ条件）
vd = pd.read_csv("2026-10-16_video_daily_views.csv", encoding="utf-8-sig", parse_dates=["日付"])
v["コンテンツ"] = v["コンテンツ"].str.strip()   # 先頭に空白が付いた動画IDをそろえる（SQL の TRIM）
vd["コンテンツ"] = vd["コンテンツ"].str.strip()
print(vd["コンテンツ"].nunique(), vd["視聴回数"].sum())   # 24本・12,742回（各上位12本だけの表）

m = vd.merge(v[["コンテンツ", "種類", "公開日"]], on="コンテンツ")   # SQL の JOIN
m["公開後日数"] = (m["日付"] - m["公開日"]).dt.days   # SQL の DATEDIFF(day, 公開日, 日付)
first7 = m[(m["公開日"] <= "2026-10-08") & m["公開後日数"].between(0, 6)]   # 7日分そろっている動画だけ

d7 = first7.groupby(["種類", "コンテンツ"])["視聴回数"].sum().reset_index()   # SQL の GROUP BY
d7 = d7.sort_values("視聴回数", ascending=False).groupby("種類").head(5)   # SQL の RANK() … WHERE RankInType <= 5
d7["タイトル"] = d7["コンテンツ"].map(v.set_index("コンテンツ")["動画のタイトル"].str[:40])
print(d7.sort_values(["種類", "視聴回数"], ascending=[True, False]).to_string(index=False))